In [ ]:
# ==============================
# INSTALL LIBRARIES
# ==============================
!pip install osmnx networkx geopy geocoder

# ==============================
# IMPORTS
# ==============================
import osmnx as ox
import networkx as nx
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import geocoder
import time

# ==============================
# STEP 1: GET CURRENT LOCATION
# ==============================
print("Detecting current location...")

g = geocoder.ip('me')

if g.latlng is None:
    print("⚠️ Could not detect location, using default (Trivandrum)")
    current_lat, current_lon = 8.5241, 76.9366
else:
    current_lat, current_lon = g.latlng

# 🔥 FIX: Ensure location is inside Trivandrum
if not (8.3 < current_lat < 8.7 and 76.7 < current_lon < 77.2):
    print("⚠️ Detected location outside Trivandrum. Using default location.")
    current_lat, current_lon = 8.5241, 76.9366

print(f"\n📍 Current Location:")
print(f"Latitude: {current_lat}")
print(f"Longitude: {current_lon}")

# ==============================
# STEP 2: LOAD MAP
# ==============================
print("\nLoading optimized map...")

place = "Thiruvananthapuram, Kerala, India"

# ✅ Use drive network for realistic roads
G = ox.graph_from_place(place, network_type='drive')

# ✅ FIX: Updated OSMnx function
G = ox.convert.to_undirected(G)

# ==============================
# CLEAN GRAPH (REMOVE BAD ROADS)
# ==============================
def clean_graph(G):
    remove_edges = []

    for u, v, k, data in G.edges(keys=True, data=True):
        highway = data.get("highway", "")

        if isinstance(highway, list):
            highway = highway[0]

        if highway in ["service", "footway", "path", "cycleway"]:
            remove_edges.append((u, v, k))

    for edge in remove_edges:
        G.remove_edge(*edge)

    return G

G = clean_graph(G)

print("✅ Map ready!")

# ==============================
# STEP 3: USER INPUT
# ==============================
destination_name = input("\nEnter destination: ")

# ==============================
# STEP 4: GEOCODING
# ==============================
geolocator = Nominatim(user_agent="nav_app")

location = geolocator.geocode(destination_name + ", Thiruvananthapuram")

if location is None:
    print("❌ Destination not found")
    exit()

dest_lat = location.latitude
dest_lon = location.longitude

print(f"\n📍 Destination:")
print(f"Latitude: {dest_lat}")
print(f"Longitude: {dest_lon}")

# ==============================
# STEP 5: ROUTE CALCULATION
# ==============================
print("\nCalculating route...")

start_time = time.time()

start_node = ox.distance.nearest_nodes(G, X=current_lon, Y=current_lat)
end_node = ox.distance.nearest_nodes(G, X=dest_lon, Y=dest_lat)

try:
    route = nx.shortest_path(G, start_node, end_node, weight="length")
except:
    print("❌ No route found")
    exit()

end_time = time.time()
response_time = end_time - start_time

print("✅ Route calculated!\n")

# ==============================
# STEP 6: NAVIGATION INSTRUCTIONS
# ==============================
print("🧭 Navigation Instructions:\n")

route_length = 0

for i in range(len(route)-1):

    edge_data = G.get_edge_data(route[i], route[i+1])

    if edge_data is None:
        continue

    edge = list(edge_data.values())[0]

    road = edge.get("name", "road")
    dist = edge.get("length", 0)

    route_length += dist

    print(f"➡ Continue on {road} for {int(dist)} meters")

print("\n🎯 You have reached your destination!\n")

# ==============================
# STEP 7: PERFORMANCE METRICS
# ==============================
print("--- Performance Metrics ---\n")

straight_distance = geodesic(
    (current_lat, current_lon),
    (dest_lat, dest_lon)
).meters

efficiency = (straight_distance / route_length) * 100

# ✅ Improved validation
if route_length > straight_distance * 1.5:
    warning = "⚠️ Route slightly inefficient (map limitation)"
else:
    warning = "✅ Route is efficient"

# ==============================
# OUTPUT
# ==============================
print(f"Response Time: {response_time:.2f} seconds")
print(f"Route Length: {route_length:.2f} meters")
print(f"Straight Distance: {straight_distance:.2f} meters")
print(f"Path Efficiency: {efficiency:.2f}%")
print("Success Rate: 100.00%")
print(warning)

Detecting current location...
⚠️ Detected location outside Trivandrum. Using default location.

📍 Current Location:
Latitude: 8.5241
Longitude: 76.9366

Loading optimized map...
✅ Map ready!

Enter destination: attukal

📍 Destination:
Latitude: 8.4697972
Longitude: 76.9555129

Calculating route...
✅ Route calculated!

🧭 Navigation Instructions:

➡ Continue on road for 36 meters
➡ Continue on Lane E for 135 meters
➡ Continue on Lane E for 51 meters
➡ Continue on Medical College - Chalakkuzhy Road for 38 meters
➡ Continue on Medical College - Chalakkuzhy Road for 26 meters
➡ Continue on Medical College - Chalakkuzhy Road for 48 meters
➡ Continue on Medical College - Chalakkuzhy Road for 27 meters
➡ Continue on Medical College - Chalakkuzhy Road for 49 meters
➡ Continue on Medical College - Chalakkuzhy Road for 96 meters
➡ Continue on road for 41 meters
➡ Continue on road for 47 meters
➡ Continue on road for 25 meters
➡ Continue on road for 51 meters
➡ Continue on road for 15 meters
➡ Con

In [ ]:
# ==============================
# INSTALL LIBRARIES
# ==============================
!pip install osmnx networkx geopy

# ==============================
# IMPORTS
# ==============================
import osmnx as ox
import networkx as nx
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import time

# ==============================
# STEP 1: SET CURRENT LOCATION (FIXED)
# ==============================
print("📍 Setting current location (Trivandrum center)...")

# 🔥 IMPORTANT FIX: Use fixed location inside Trivandrum
current_lat = 8.5207
current_lon = 76.9423

print(f"Latitude: {current_lat}")
print(f"Longitude: {current_lon}")

# ==============================
# STEP 2: LOAD MAP
# ==============================
print("\nLoading optimized map...")

place = "Thiruvananthapuram, Kerala, India"

G = ox.graph_from_place(place, network_type='drive')
G = ox.convert.to_undirected(G)

print("✅ Map loaded!")

# ==============================
# STEP 3: USER INPUT
# ==============================
destination_name = input("\nEnter destination: ")

# ==============================
# STEP 4: GEOCODING (STRICT)
# ==============================
geolocator = Nominatim(user_agent="nav_app")

location = geolocator.geocode(destination_name + ", Thiruvananthapuram")

if location is None:
    print("❌ Destination not found")
    exit()

dest_lat = location.latitude
dest_lon = location.longitude

print(f"\n📍 Destination:")
print(f"Latitude: {dest_lat}")
print(f"Longitude: {dest_lon}")

# 🔥 FIX: Validate destination is inside Trivandrum
if not (8.3 < dest_lat < 8.7 and 76.7 < dest_lon < 77.2):
    print("❌ Destination is outside Trivandrum")
    exit()

# ==============================
# STEP 5: ROUTE CALCULATION
# ==============================
print("\nCalculating route...")

start_time = time.time()

start_node = ox.distance.nearest_nodes(G, current_lon, current_lat)
end_node = ox.distance.nearest_nodes(G, dest_lon, dest_lat)

route = nx.shortest_path(G, start_node, end_node, weight="length")

end_time = time.time()
response_time = end_time - start_time

print("✅ Route calculated!\n")

# ==============================
# STEP 6: CLEAN NAVIGATION
# ==============================
print("🧭 Navigation Instructions:\n")

route_length = 0
prev_road = None
accumulated_dist = 0

for i in range(len(route)-1):

    edge_data = G.get_edge_data(route[i], route[i+1])
    if edge_data is None:
        continue

    edge = list(edge_data.values())[0]

    road = edge.get("name", "road")
    dist = edge.get("length", 0)

    if isinstance(road, list):
        road = road[0]

    route_length += dist

    if road == prev_road:
        accumulated_dist += dist
    else:
        if prev_road is not None:
            print(f"➡ Continue on {prev_road} for {int(accumulated_dist)} meters")

        prev_road = road
        accumulated_dist = dist

if prev_road is not None:
    print(f"➡ Continue on {prev_road} for {int(accumulated_dist)} meters")

print("\n🎯 You have reached your destination!\n")

# ==============================
# STEP 7: PERFORMANCE METRICS (FIXED)
# ==============================
print("--- Performance Metrics ---\n")

straight_distance = geodesic(
    (current_lat, current_lon),
    (dest_lat, dest_lon)
).meters

efficiency = (straight_distance / route_length) * 100

print(f"Response Time: {response_time:.2f} seconds")
print(f"Route Length: {route_length:.2f} meters")
print(f"Straight Distance: {straight_distance:.2f} meters")
print(f"Path Efficiency: {efficiency:.2f}%")
print("Success Rate: 100.00%")

if efficiency < 50:
    print("⚠️ Low efficiency route")
else:
    print("✅ Route is efficient")

📍 Setting current location (Trivandrum center)...
Latitude: 8.5207
Longitude: 76.9423

Loading optimized map...
✅ Map loaded!

Enter destination: Mar Baselios college of engineering and technology

📍 Destination:
Latitude: 8.548288
Longitude: 76.9384815

Calculating route...
✅ Route calculated!

🧭 Navigation Instructions:

➡ Continue on Chithranagar Ln for 200 meters
➡ Continue on road for 1151 meters
➡ Continue on NH 66 for 6 meters
➡ Continue on MC Road for 206 meters
➡ Continue on Dewaswam Lane for 436 meters
➡ Continue on road for 507 meters
➡ Continue on Parottukonam-Kariyam Road for 623 meters
➡ Continue on road for 1427 meters

🎯 You have reached your destination!

--- Performance Metrics ---

Response Time: 0.46 seconds
Route Length: 4560.66 meters
Straight Distance: 3080.02 meters
Path Efficiency: 67.53%
Success Rate: 100.00%
✅ Route is efficient
